### Определение орбиты виртуального астероида методом Лапласа

In [1]:
from math import *
import numpy as np
from datetime import datetime
from astropy.time import Time

from scipy.optimize import dual_annealing
from scipy.optimize import minimize

import matplotlib.pyplot as plt
%matplotlib widget

### Матрицы вращения

In [2]:
def rotx(psi):
    return np.array([[1,0,0],[0,cos(psi),sin(psi)],[0,-sin(psi),cos(psi)]])

def roty(psi):
    return np.array([[cos(psi),0,-sin(psi)],[0,1,0],[sin(psi),0,cos(psi)]])

def rotz(psi):
    return np.array([[cos(psi),sin(psi),0],[-sin(psi),cos(psi),0],[0,0,1]])


### Теперь нужна функция для перехода от координат и скоростей к элементам орбиты

In [3]:
def orbital_elem(x,y,z,xdot,ydot,zdot):
    r = np.array([x,y,z])
    rdot = np.array([xdot,ydot,zdot])
    #modules of r and rdot vectors
    R = sqrt(x*x+y*y+z*z)
    vr = np.dot(r/R,rdot)
#     RDOT = sqrt(xdot*xdot+ydot*ydot+zdot*zdot)
    #double sectorial velocity
    c = np.array([y*zdot-z*ydot,z*xdot - x*zdot,x*ydot-y*xdot])
    C = np.linalg.norm(c)
#     a = 1 / (2/R-RDOT*RDOT) # semi-major axis
    p = C**2 # focal parameter
    
    orb1 = p/R-1.0
    orb2 = vr*C
    
    e = sqrt(orb1*orb1+orb2*orb2)
    a = p/(1-e*e)

#     e = sqrt(1-p/a) # eccentricity
    u = atan2(orb2/e,orb1/e) # true anomaly
    # eccentric anomaly
    E = atan2(R*sin(u)/a/sqrt(1-e**2),R*cos(u)/a+e)
    M = E - e*sin(E) # mean anomaly
    P = 2*pi*a**(3/2) # period
    n = a**(-3/2) # mean motion
    T = - M / n # perihelion epoch
    #inclination
    i = atan2(c[2]/C,sqrt(c[0]**2+c[1]**2)/C)
    # longitude of the ascending node
    N = np.cross(np.array([0,0,1]),c/C)
#     print(N)
    Omega = atan2(N[1],N[0])
    omega_plus_u = atan2(np.dot(np.cross(N,r/R),c/C),np.dot(r/R,N))
    omega = omega_plus_u - u # argument of periapsis
    return a,e,P,T,i,Omega,omega

### Нам надо уметь решать уравнение Кеплера

In [4]:
def KeplerEquation(e, P, T, t):
    n = 2 * pi / P
    E = M = n * (t - T)
    epsilon = pi/(180*3600000)
    while (fabs(E - e * sin(E) - M) > epsilon):
        E = M + e * sin(E)
    return E

### А еще нам надо уметь вычислять компоненты гелиоцентрического радиуса-вектора если задан момент времени $t$ и орбитальные элементы

In [5]:
def xyz(a,e,P,T,i,Omega, omega,t):   # defining heliocentric coordinates
    E = KeplerEquation(e,P,T,t)
    ksi = a*(cos(E)-e)
    eta = a*sqrt(1-e**2)*sin(E)
    r = np.array([ksi, eta, 0])
#     print(np.linalg.norm(r))
#     return np.linalg.multi_dot([rotz(-Omega),rotx(-i),rotz(-omega),r])
    return np.dot(rotz(-Omega),np.dot(rotx(-i),np.dot(rotz(-omega),r)))

def dot_xyz(a,e,P,T,i,Omega, omega,t):   # defining heliocentric coordinates
    E = KeplerEquation(e,P,T,t)
    dotE = 2 * pi / P /(1-e*cos(E))
    dotx = -sin(E)*dotE
    doty = sqrt(1-e**2)*cos(E)*dotE
    dotr = np.array([dotx, doty, 0])
    return np.dot(rotz(-Omega),np.dot(rotx(-i),np.dot(rotz(-omega),a*dotr)))

def ddot_xyz(a,e,P,T,i,Omega, omega,t):   # defining heliocentric coordinates
    E = KeplerEquation(e,P,T,t)
    n = 2 * pi / P
    dotE =  n/(1-e*cos(E))
    ddotE = -e*n*sin(E)*dotE/(1-e*cos(E))**2
    
    ddotx = -cos(E)*dotE*dotE-sin(E)*ddotE
    ddoty = sqrt(1-e**2)*(cos(E)*ddotE-sin(E)*dotE*dotE)
    ddotr = np.array([ddotx, ddoty, 0])
    return np.dot(rotz(-Omega),np.dot(rotx(-i),np.dot(rotz(-omega),a*ddotr)))

def xyz0(r0,n0,t,t0):# Geocenter heliocentric coordinates
    return np.array([r0*cos(n0*(t-t0)),r0*sin(n0*(t-t0)),0])
def dot_xyz0(r0,n0,t,t0):# Geocenter heliocentric coordinates
    return np.array([-r0*sin(n0*(t-t0)),r0*cos(n0*(t-t0)),0])
def ddot_xyz0(r0,n0,t,t0):# Geocenter heliocentric coordinates
    return np.array([-r0*cos(n0*(t-t0)),-r0*sin(n0*(t-t0)),0])

### Гелиоцентрические положение и скорость центра Земли на какой-то произвольный момент введем как показано ниже. Для координат единицей измерения является астрономическая единица, а для скоростей за единицу принимается круговая скорость движения Земли по гелиоцентрической орбите ($V \approx 29844.9$~м/с).

In [6]:
x0 = 1; y0 = 0; z0 =0; x0dot = 0; y0dot = 1; z0dot = 0;
speed_of_light = 10045# в скоростях Земли

### Для тестирования мы поработаем с неким модельным астероидом. Его орбита определяется декартовыми гелиоцентрическими координатами и их первыми производными по времени (компонентами скорости)

In [7]:
x = -4.42886945; y = 0.48312824; z = 0.48312824 # au
xdot = -0.30709289; ydot = -0.27567732; zdot = -0.27567732 # в скоростях Земли

In [8]:
a,e,P,T,i,Omega,omega = orbital_elem(x,y,z,xdot,ydot,zdot)
print(a,e,P,T,i,Omega,omega)
xyz(a,e,P,T,i,Omega, omega,0)
dot_xyz(a,e,P,T,i,Omega, omega,0)

4.999979434670496 0.4999994190489909 70.24771390796397 -9.75670635217849 0.7853981633974483 0.0 1.0852708315703343


array([-0.30709289, -0.27567732, -0.27567732])

#### Моделируем наблюдения

In [9]:
delta_t = 0.05 # шаг по времени между наблюдениями
a,e,P,T,i,Omega,omega = orbital_elem(x,y,z,xdot,ydot,zdot)

N = 8

lon,lat,tau = np.zeros(N),np.zeros(N),np.zeros(N)

for k in range(N):
    t = 1+k*delta_t  #observational time
    t_real = t - np.linalg.norm(xyz(a,e,P,T,i,Omega, omega,t)-xyz0(1,1,t,0))/speed_of_light# планетная аберрация
#     print(t_real-t)
    xyz_real = xyz(a,e,P,T,i,Omega, omega,t_real)-xyz0(1,1,t,0)# геоцентрическое положение
    rho = np.linalg.norm(xyz_real)
#     print(rho,np.linalg.norm(xyz(a,e,P,T,i,Omega, omega,t_real)))
    urho = xyz_real/rho
    b = atan2(urho[2],sqrt(urho[0]**2+urho[1]**2))
    l = atan2(urho[1]/cos(b),urho[0]/cos(b))
    tau[k] = t
    lat[k] = b
    lon[k] = l
    print('time = %5.3f'%t,'lon = %5.3f'%degrees(l),'lat = %5.4f'%degrees(b))
lat+=np.random.normal(0,radians(0.001/3600),N)
lon+=np.random.normal(0,radians(0.001/3600),N)

time = 1.000 lon = -173.096 lat = 2.2246
time = 1.050 lon = -172.626 lat = 2.0833
time = 1.100 lon = -172.173 lat = 1.9410
time = 1.150 lon = -171.738 lat = 1.7977
time = 1.200 lon = -171.321 lat = 1.6531
time = 1.250 lon = -170.923 lat = 1.5072
time = 1.300 lon = -170.545 lat = 1.3599
time = 1.350 lon = -170.188 lat = 1.2110


In [10]:
fig,ax = plt.subplots(1,figsize=(1.5, 1.5), dpi=300)

ax.scatter(np.degrees(lon),np.degrees(lat),  c='black',  s=0.3, linewidths=0.5)
ax.set_xlabel('$\\lambda$ (deg)', fontsize = 6)
ax.set_ylabel('$\\beta$ (deg)', fontsize = 6)

ax.tick_params(axis='both', which='major', labelsize=4)
ax.tick_params(axis='both', which='minor', labelsize=4)
ax.ticklabel_format(useOffset=False)

plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

#### Определяем параметры видимого движения с помощью МНК

$\lambda = \lambda(t_0)+\dot \lambda(t-t_0)+0.5\ddot \lambda (t-t_0)^2$

In [11]:
C = np.ones((N,3))
meant = np.mean(tau)
for j in range(N):
    C[j,1]=tau[j]- meant
    C[j,2]=0.5*(tau[j]- meant)**2

Lat = np.dot(np.linalg.inv(np.dot(C.T,C)),np.dot(C.T,lat))
Lon = np.dot(np.linalg.inv(np.dot(C.T,C)),np.dot(C.T,lon))

res = lat-np.dot(C,Lat)
uwe = np.dot(res.T,res)/(N-3)
D = uwe*np.linalg.inv(np.dot(C.T,C))
print(Lon,Lat)
print(np.degrees(Lat),np.degrees(Lon))
3600*np.degrees(res),3600*degrees(sqrt(uwe))

[-2.99370892  0.14513483 -0.13151675] [ 0.03011636 -0.05052283 -0.00877892]
[ 1.72554028 -2.89474488 -0.50299506] [-171.52688597    8.31561336   -7.53535457]


(array([ 0.82499898, -0.59272213, -0.82326965, -0.34917748,  0.35762579,
         0.82073738,  0.58004597, -0.81823886]),
 0.8531254180114897)

Итак, выпишем уравнения в том виде, как они представлены в презентации

$$
\begin{matrix}
-k^2 (X+\lambda\rho) r^{-3}&=&\ddot{X}+\ddot{\lambda}\rho+2\dot{\lambda}\dot{\rho}+\lambda\ddot{\rho}\\
-k^2 (Y+\mu\rho) r^{-3}&=&\ddot{Y}+\ddot{\mu}\rho+2\dot{\mu}\dot{\rho}+\mu\ddot{\rho}\\
-k^2 (Z+\nu\rho) r^{-3}&=&\ddot{Z}+\ddot{\nu}\rho+2\dot{\nu}\dot{\rho}+\nu\ddot{\rho}\\
r^2& = &\rho^2+R^2-2\rho R\cos(l)
\end{matrix}
$$ 

Блок кода ниже содержит представление $X,Y,Z$, $\dot X,\dot Y,\dot Z$, $\ddot X,\ddot Y,\ddot Z$, которые вычисляются через соответствующие функции.

Кроме того, здесь все направляющие косинусы, их первые и вторые производные по времени с учетей аппроксимации данных наблюдений

In [12]:
# переобозначение для удобства восприятия
l = Lon[0];dotl = Lon[1];ddotl = Lon[2]
b = Lat[0];dotb = Lat[1];ddotb = Lat[2]

# положение и ускорение геоцентра
X,Y,Z = xyz0(1,1,meant,0)
R = sqrt(X*X+Y*Y+Z*Z)
dotX,dotY,dotZ = dot_xyz0(1,1,meant,0)
ddotX,ddotY,ddotZ = ddot_xyz0(1,1,meant,0)

# направляющие косинусы
la,mu,nu = cos(l)*cos(b),sin(l)*cos(b),sin(b)
# первые производные направляющих косинусов
dot_la = - sin(b)*cos(l)*dotb - sin(l)*cos(b)*dotl
dot_mu = - sin(b)*sin(l)*dotb + cos(b)*cos(l)*dotl
dot_nu = cos(b)*dotb
# вторые производные направляющих косинусов    
ddot_la,ddot_mu,ddot_nu = ddotl*np.array([sin(l)*cos(b),cos(l)*cos(b),0])+\
            ddotb*np.array([cos(l)*sin(b),-sin(l)*sin(b),cos(b)])-\
          (dotl**2)*np.array([-cos(l)*cos(b),sin(l)*cos(b),0])-\
            (dotb**2)*np.array([-cos(l)*cos(b),sin(l)*cos(b),sin(b)])+\
            2*(dotl*dotb)*np.array([-sin(l)*sin(b),-cos(l)*sin(b),0])


Мы видели, что решение уравнений метода Лапласа является сложной задачей. Легко ошибиться в наборе кода, сложно понимать и контролировать смысл действий (введения новых переменных и решения уравнения восьмой степени). Поэтому во многих публикациях встречается некая итеративная процедура подбора оптимального решения.

В нашем примере предлагается следующая схема.
1. Мы знаем, что имеем дело с астероидом. Логично просто образовать массив значений $\rho$ от 0.01 а.е. до 10 а.е.
2. Сообразно $\rho$, несложно подсчитать $r$.
3. Теперь в наших уравнениях из четырех неизвестных для каждого $k$-того уравнения остается две неизвестных, которые можно вычислить по первым двум уравнениям.

$$
\begin{matrix}
-k^2 (X+\lambda\rho) r^{-3}-\ddot{X}-\ddot{\lambda}\rho+&=&2\dot{\lambda}\dot{\rho}+\lambda\ddot{\rho}\\
-k^2 (Y+\mu\rho) r^{-3}-\ddot{Y}-\ddot{\mu}\rho&=&2\dot{\mu}\dot{\rho}+\mu\ddot{\rho}\\
\end{matrix}
$$ 

То есть, для каждой пары $\rho$, $r$ получаем $\dot{\rho},\ddot{\rho}$.

Теперь можно вычислить, насколько хорошо эти оценки $\dot{\rho},\ddot{\rho}$ удовлетворяют третьему уравнению.

$$
\begin{matrix}
D = |-k^2 (Z+\nu\rho) r^{-3}&-&(\ddot{Z}+\ddot{\nu}\rho+2\dot{\nu}\dot{\rho}+\nu\ddot{\rho})|\\
\end{matrix}
$$

То есть мы соберем в массив все разности, образованные из третьего уравнения. Логично ожидать, что для наиболее вероятной пары $\rho$, $r$, разность $D$ будет минимальной из всех возможных. Таким образом мы сможем найти приближение, которое можно использовать в качестве предварительного результата


In [13]:
N = 20000
    
rho  = np.linspace(0.01,10,N)
r = np.sqrt(rho**2 +R*R-2*rho*(-X*la-Y*mu-Z*nu))

M = np.array([
        [2*dot_la,la],
        [2*dot_mu,mu]
    ])

res = np.zeros((N,3))
for k in range(N):
    L = np.array([
        -(X+la*rho[k])/(r[k]**3)-ddotX-ddot_la*rho[k],
        -(Y+mu*rho[k])/(r[k]**3)-ddotY-ddot_mu*rho[k]
    ])
    D = np.dot(np.linalg.inv(M),L)
    res[k,0] = fabs(-(Z+nu*rho[k])/(r[k]**3)-ddotZ-ddot_nu*rho[k]-2*dot_nu*D[0]-nu*D[1])
    res[k,1] = D[0]
    res[k,2] = D[1]
    
minn = np.argmin(res[:,0])
dot_rho = res[minn,1]
ddot_rho = res[minn,2]
print(r[minn],rho[minn],dot_rho)
    
fig,ax = plt.subplots(1,figsize=(2.5, 1.2), dpi=300)
ax.scatter(r,res[:,0],  c='blue',  s=0.1, linewidths=0.0)
ax.scatter(rho,res[:,0],  c='red',  s=0.1, linewidths=0.0)
ax.set_xlabel('r,$\\rho$', fontsize = 6)
ax.set_ylabel('$\\Delta$ ', fontsize = 6)
# ax.set_ylim(-10,10)
ax.tick_params(axis='both', which='major', labelsize=4)
ax.tick_params(axis='both', which='minor', labelsize=4)
ax.ticklabel_format(useOffset=False)
plt.tight_layout()

fig,ax = plt.subplots(1,figsize=(2.5, 1.2), dpi=300)
ax.scatter(r,res[:,1],  c='green',  s=0.1, linewidths=0.0)
ax.scatter(r,res[:,2],  c='blue',  s=0.1, linewidths=0.0)
ax.set_xlabel('r', fontsize = 6)
ax.set_ylabel('$\\dot\\rho,\\ddot\\rho$ ', fontsize = 6)
# ax.set_ylim(-10,10)
ax.tick_params(axis='both', which='major', labelsize=4)
ax.tick_params(axis='both', which='minor', labelsize=4)
ax.ticklabel_format(useOffset=False)
plt.tight_layout()

4.705567983621209 5.144117705885295 -0.5787043254290042


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Мы действительно получили некие величины $r,\rho,\dot\rho,\ddot\rho$, которые делают строгими первые два уравнения и оставляют небольшое отличие левой и правой частей третьего уравнения. Учитывая, что наши наблюдения характеризуются случайными ошибками, логично подобрать какой-то оптимальный набор $r,\rho,\dot\rho,\ddot\rho$, минимизирующий разности левых и правых частей для всех трех уравнений так, чтобы ошибка распределялась более равномерно между этими уравнениями.

Поэтому мы строим функцию Laplace(p), которая выдает норму вектора разностей для наших уравнений. Далее мы выполнеяем минимизацию по схеме Нелдера-Мида. Наши предварительные $r,\rho,\dot\rho,\ddot\rho$ используются как нуль-приближение.

In [14]:
def Laplace(p):
    r,rho,dot_rho,ddot_rho = p
    DD=np.array([
        -(X+la*rho)/(r**3)-(ddotX+ddot_la*rho+2*dot_la*dot_rho+la*ddot_rho),
        -(Y+mu*rho)/(r**3)-(ddotY+ddot_mu*rho+2*dot_mu*dot_rho+mu*ddot_rho),
        -(Z+nu*rho)/(r**3)-(ddotZ+ddot_nu*rho+2*dot_nu*dot_rho+nu*ddot_rho),
    ])
    return np.linalg.norm(DD)

lw = [r[minn],rho[minn],dot_rho,ddot_rho]# нижняя граница 
up = [r[minn],rho[minn],dot_rho,ddot_rho]# верхняя граница  bounds=list(zip(lw, up)),
res = minimize(Laplace, x0 = np.array([r[minn],rho[minn],dot_rho,ddot_rho]),  method='Nelder-Mead', tol=1e-3)
print(r[minn],rho[minn],dot_rho,ddot_rho)
res

4.705567983621209 5.144117705885295 -0.5787043254290042 -0.4785460104780628


 final_simplex: (array([[ 4.71775763,  5.14433727, -0.57882269, -0.47820143],
       [ 4.71856843,  5.14436395, -0.57882473, -0.47817755],
       [ 4.71819996,  5.1443616 , -0.57882164, -0.47818873],
       [ 4.71677873,  5.14432236, -0.57881266, -0.47822851],
       [ 4.71773674,  5.14433526, -0.57882574, -0.47820126]]), array([5.08500814e-07, 5.70192837e-07, 6.33912781e-07, 7.15670612e-07,
       7.65725272e-07]))
           fun: 5.085008141317449e-07
       message: 'Optimization terminated successfully.'
          nfev: 116
           nit: 62
        status: 0
       success: True
             x: array([ 4.71775763,  5.14433727, -0.57882269, -0.47820143])

любопытно посмотреть наши разности. В блоке выше параметр fun - это и есть норма разностей.

Код ниже показывает, что теперь оценки $r,\rho,\dot\rho,\ddot\rho$ распределяют разности частей уравнений более равномерно.

In [15]:
r,rho,dot_rho,ddot_rho = res.x
print(-(X+la*rho)/(r**3)-(ddotX+ddot_la*rho+2*dot_la*dot_rho+la*ddot_rho),
        -(Y+mu*rho)/(r**3)-(ddotY+ddot_mu*rho+2*dot_mu*dot_rho+mu*ddot_rho),
        -(Z+nu*rho)/(r**3)-(ddotZ+ddot_nu*rho+2*dot_nu*dot_rho+nu*ddot_rho))

-7.075789567462243e-08 -3.014415894089968e-07 4.254787295532563e-07


Вот здесь мы формируем гелиоцентрический вектор состояния согласно нашим уравнеиям

In [16]:
r,rho,dot_rho,ddot_rho = res.x
x_ = X+la*rho
y_ = Y+mu*rho
z_ = Z+nu*rho
xdot_ = dotX+dot_la*rho+la*dot_rho
ydot_ = dotY+dot_mu*rho+mu*dot_rho
zdot_ = dotZ+dot_nu*rho+nu*dot_rho

Сравниваем параметры модельной орбиты и результаты наших вычислений. Можно попытаться посмотреть, какими свойствами должен обладать ряд наблюдений, чтобы сходимость была наилучшей

In [17]:
a,e,P,T,i,Omega,omega = orbital_elem(x,y,z,xdot,ydot,zdot)
print(a,e,P,T,i,Omega,omega)
print()
a_,e_,P_,T_,i_,Omega_,omega_ = orbital_elem(x_,y_,z_,xdot_,ydot_,zdot_)
print(a_,e_,P_,T_,i_,Omega_,omega_)


4.999979434670496 0.4999994190489909 70.24771390796397 -9.75670635217849 0.7853981633974483 0.0 1.0852708315703343

4.663029283303232 0.4999859148584986 63.26770522934696 -10.967829214525507 0.7708510676845937 -0.003101654229477589 0.9855027502828326


<b>Задание.</b> Допустим в блоке кода "Моделируем наблюдения" параметры delta_t = 0.035,  N = 20.
Протестируйте работу метода. Почему орбитальные параметры определяются хуже? Ведь наблюдения охватывают большую дугу и производные должны вычисляться точнее. Попробуйте изменить код так, чтобы данная проблема успешно решалась. 

In [18]:
# x_ = X+la*rho[minn]
# y_ = Y+mu*rho[minn]
# z_ = Z+nu*rho[minn]
# xdot_ = dotX+dot_la*rho[minn]+la*dot_rho
# ydot_ = dotY+dot_mu*rho[minn]+mu*dot_rho
# zdot_ = dotZ+dot_nu*rho[minn]+nu*dot_rho